# SmolVLM Inference

In [ ]:
!pip install -q transformers trl==0.12.1 datasets bitsandbytes peft accelerate
!pip install num2words
# transformers==4.46.3, trl==0.12.1, datasets==3.1.0, bitsandbytes==0.45.0, peft==0.13.2, accelerate==1.1.1
#!pip install -q flash-attn --no-build-isolation

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import torch.multiprocessing as mp
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

mp.set_start_method('spawn')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## 1. Testing the Fine-Tuned Model

In [ ]:
model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map='cuda',
    dtype=torch.bfloat16,
    _attn_implementation='eager', # Use `flash_attention_2` on Ampere GPUs and above and `eager` on older GPUs.
)

processor = AutoProcessor.from_pretrained(model_id)

### 1.1. Generate text from Image and Message
1. Pass image as RGB image
2. Message:
    - Using `apply_chat_template`
    - The input to apply_chat_template should be structured as a list of dictionaries with `role` and `content` keys.
    - The common roles are:
        - `system` for directives on how the model should act (placed at the beginning)
        - `user` for messages from the user
        - `assistant` for messages from the model
    - `apply_chat_template` takes this list and returns a formatted sequence

In [ ]:
def generate_text_from_sample(model, processor, image, message, max_new_tokens=1024, device='cuda'):
    # Prepare the text input
    text_input = processor.apply_chat_template(
        message,  # Use the user message
        add_generation_prompt=True
    )

    # Display the text
    #print(text_input)

    # Prepare the image input
    image = Image.open(image).convert('RGB')
    image_inputs = []
    image_inputs.append([image]) # convert to list as processor requires it

    # Display the image
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Prepare the inputs for the model
    model_inputs = processor(
        text=text_input,
        images=image_inputs,
        return_tensors='pt', # Return PyTorch tensors
    ).to(device, dtype=torch.bfloat16)

    # Generate text with the model
    generated_token_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated token ids to remove the input token ids
    # Remove the original input tokens from the generated sequence
    trimmed_generated_token_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_token_ids)
    ]

    # Decode the output text
    # This converts the generated token IDs back into human-readable text
    output_text = processor.batch_decode(
        trimmed_generated_token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]

### 1.2. Test on unseen image

In [ ]:
test_image = 'test-3.jpg'
user_message = 'OCR this image accurately.'
message = [{'role': 'user','content': [
                {'type': 'image'},
                {'type': 'text', 'text': user_message}]},]
output = generate_text_from_sample(model, processor,test_image, message)
print(output)